In [1]:
import torch
from torch import nn
from torch.nn import functional as F

In [3]:
class RNNScratch(nn.Module):
    def __init__(self, num_inputs, num_hiddens, sigma=0.01):
        super().__init__()
        self.W_xh = nn.Parameter(torch.randn(num_inputs, num_hiddens) * sigma)
        self.W_hh = nn.Parameter(torch.randn(num_hiddens, num_hiddens) * sigma)
        self.b_h = nn.Parameter(torch.zeros(num_hiddens))

        self.num_hiddens = num_hiddens
        self.num_inputs = num_inputs
        self.sigma = sigma

    def forward(self, inputs, state=None):
        if state is None:
            state = torch.zeros(
                (inputs.shape[1], self.num_hiddens), device=inputs.device
            )
        outputs = []
        for X in inputs:
            state = torch.tanh(X @ self.W_xh + state @ self.W_hh + self.b_h)
            outputs.append(state)
        return outputs, state

In [4]:
F.one_hot(torch.tensor([0, 2]), num_classes=5)

tensor([[1, 0, 0, 0, 0],
        [0, 0, 1, 0, 0]])

In [6]:
def one_hot(self, X):
    return F.one_hot(X.T, self.vocab_size).type(torch.float32)

In [7]:
def output_layer(self, rnn_outputs):
    outputs = [H @ self.W_hq + self.b_q for H in rnn_outputs]
    return torch.stack(outputs, dim=1)

In [8]:
class RNNLM(nn.Module):
    def __init__(self, rnn, vocab_size, lr=1e-3):
        super().__init__()

        self.rnn = rnn
        self.vocab_size = vocab_size
        self.lr = lr

        self.W_hq = nn.Parameter(torch.randn(rnn.num_hiddens, vocab_size) * rnn.sigma)
        self.b_q = nn.Parameter(torch.zeros(vocab_size))

    def one_hot(self, X):
        return F.one_hot(X.T, self.vocab_size).type(torch.float32)

    def output_layer(self, rnn_outputs):
        outputs = [H @ self.W_hq + self.b_q for H in rnn_outputs]
        return torch.stack(outputs, dim=1)

    def forward(self, X, state=None):
        embs = self.one_hot(X)
        rnn_outputs, state = self.rnn(embs, state)
        return self.output_layer(rnn_outputs)
    

In [9]:
batch_size, num_inputs, num_hiddens, num_steps = 32, 100, 256, 100

rnn = RNNScratch(num_inputs, num_hiddens)
model = RNNLM(rnn, vocab_size=num_inputs)

X = torch.ones((batch_size, num_steps), dtype=torch.int64)
outputs = model(X)

### Gradient Clipping

> Dùng để tránh hiện tượng exploding gradient

> Cách hoạt động: Nếu độ lớn của gradient vượt quá một ngưỡng nhất định, ta sẽ chia tất cả các giá trị của gradient cho một số sao cho độ lớn của gradient bằng với ngưỡng đó. Điều này giúp giữ cho các giá trị của gradient không quá lớn và giúp mô hình học tốt hơn.

In [11]:
def clip_gradients(model, grad_clip_val):
    """
    Global norm: Nối gradient của tất cả params thành 1 vector duy nhất
    -> tính 1 norm duy nhất
    -> nếu norm > grad_clip_val, scale tất cả gradient về grad_clip_val / norm
    """

    params = [p for p in model.parameters() if p.requires_grad]

    norm = torch.sqrt(sum(torch.sum(p.grad**2) for p in params))
    if norm > grad_clip_val:
        for param in params:
            param.grad[:] *= grad_clip_val / norm

## Training

In [ ]:
embs = self.one_hot(X)  # 1 hot encode

rnn_outputs, state = self.rnn(embs, state)  # rnn forward

logits = self.output_layer(rnn_outputs)  # batch_size x num_steps x vocab_size

loss = F.cross_entropy(logits.reshape(-1, vocab_size), Y.T.reshape(-1))

ppl = torch.exp(loss)

loss.backward()
clip_gradients(model, 1)
optimizer.step()